# <a id='toc1_'></a>[Adaptive Prompt Elicitation — Demo](#toc0_)

**Paper:** [*Adaptive Prompt Elicitation for Text-to-Image Generation*](https://e-wxy.github.io/Adaptive-Prompt-Elicitation/) — ACM IUI 2026  


**Table of contents**<a id='toc0_'></a>    
- [Setup](#toc1_1_)    
- [Start a project](#toc1_2_)    
- [Interactive elicitation loop](#toc1_3_)    
- [Direct requirement editing](#toc1_4_)    
- [Save session](#toc1_5_)    
- [Resume a previous session](#toc1_6_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_1_'></a>[Setup](#toc0_)

**Configure API Keys:**
1. Copy `.env.example` → `.env` and fill in your API keys.
2. Install dependencies: `pip install -r requirements.txt`

In [1]:
from dotenv import load_dotenv
from IPython.display import HTML, Image as IPImage, display

from ape import Project

load_dotenv()

True

This code repository supports multiple LLM and image generation backends. The default configuration uses OpenAI (GPT-4.1-mini) for language and FAL (FLUX.1-schnell) for image generation.

Full model ID lists and provider-specific options are in [`ape/utils.py`](./ape/utils.py).

In [2]:
# LLM provider — used for query generation and requirement extraction
LLM_PROVIDER = "openai"          # "openai" | "anthropic" | "google" | "vertex"
LLM_MODEL    = "gpt-4.1-mini"

# Image generation provider — used to render generated images
IMG_PROVIDER = "fal"             # "fal" | "openai"
IMG_MODEL    = "fal-ai/flux/schnell"

**Query strategies**

| Strategy | Key | Description |
|---|---|---|
| **APE** | `"APE"` | Generates queries grounded in an information-theoretic framework (proposed method). |
| **In-Context Query** | `"In-Context"` | Leverages LLM in-context reasoning to generate queries. Recommended for token efficiency. |
| Vanilla | `"Vanilla"` | A baseline approach that generates queries without incorporating uncertainty estimation. |
| APE-MC | `"APE-MC"` | APE with Monte Carlo estimation. Theoretically more rigorous, but yields performance similar to standard APE at a higher token cost in our pilot study. |

In [3]:
# Query strategy
QUESTIONER = "APE"

## <a id='toc1_2_'></a>[Start a project](#toc0_)

Provide an image generation task and an initial description. APE extracts an initial set of visual feature requirements from the description and generates a first image.

In [4]:
project = Project(
    task="Postcard",
    description="A postcard image for a trip to Finland.",
    questioner=QUESTIONER,
    llm_provider=LLM_PROVIDER,
    llm_model=LLM_MODEL,
    img_provider=IMG_PROVIDER,
    img_model=IMG_MODEL,
    verbose=True,   # regenerate image after each visual feature update
    seed=123
)

 Initialize Project ...
 Init Requirements ...
 Image Prompt ...
 Generate Image ...


In [5]:
# View the initial intent modelling and generated image
display(HTML(project.display_requirements()))
display(IPImage(url=project.img_url, width=512))

## <a id='toc1_3_'></a>[Interactive elicitation loop](#toc0_)

Answer each question by typing your response. Repeat for as many rounds as you like.

In [6]:
for round in range(3):
    # Generate a visual query — each option is illustrated with an exemplar image
    visual_q = project.ask_visual_questions(N=5, M=4)
    display(HTML(project.display_visual_questions(-1)))
    # # Or generate a natural language query instead
    # question = project.ask_questions(N=5, M=5, json_format=False)
    # print(question)

    # Record the user's answer to the query, extract updated requirements, and regenerate the image
    answer = input(f"{project.last_question}: ")
    project.record_answer(answer)
    project.extract_requirements()

    display(HTML(project.display_requirements()))
    display(IPImage(url=project.img_url, width=512))

 Question Candidates ...
 Ask Question ...


 Answer ...
 Extract Requirements ...
 Image Prompt ...
 Generate Image ...


 Question Candidates ...
 Ask Question ...


 Answer ...
 Extract Requirements ...
 Image Prompt ...
 Generate Image ...


 Question Candidates ...
 Ask Question ...


 Answer ...
 Extract Requirements ...
 Image Prompt ...
 Generate Image ...


## <a id='toc1_4_'></a>[Direct requirement editing](#toc0_)

Users can add or modify feature requirements directly at any point.

In [7]:
project.add_requirement("The lake is surrounded by pine trees covered with snow. The image is in an artistic painterly style.")

display(HTML(project.display_requirements()))
display(IPImage(url=project.img_url, width=512))

 Add Requirement ...
 Update Requirement ...
 Image Prompt ...
 Generate Image ...


## <a id='toc1_5_'></a>[Save session](#toc0_)

Sessions are automatically logged to `outputs/logger/` in JSONL format. `save_final()` writes a summary entry and exports a JSON array. `save_images()` downloads all generated images.

In [ ]:
project.save_final()
project.save_images(save_folder="outputs/images/")

print(f"Session log: {project.log_file}")
print(f"\nFinal image prompt:\n{project.image_prompt}")

## <a id='toc1_6_'></a>[Resume a previous session](#toc0_)

Pass a `log_file` path to restore a saved session — requirements, image prompt, and conversation history are all restored.

In [ ]:
# Uncomment and set the path to resume a saved session:
# project = Project(
#     log_file="outputs/logger/<timestamp>_<task>_<project_name>_<project_id>.log",
#     questioner=QUESTIONER,
#     llm_provider=LLM_PROVIDER,
#     img_provider=IMG_PROVIDER,
# )
# display(HTML(project.display_requirements()))
# display(IPImage(url=project.img_url, width=512))